In [ ]:
import numpy as np
import pandas as pd
db = pd.read_csv(
    "../data/raw/train_transaction.csv"
)

identity = pd.read_csv(
    "../data/raw/train_identity.csv"
)

print("Transaction:", db.shape)
print("Identity:", identity.shape)

Transaction: (590540, 394)
Identity: (144233, 41)


In [1]:
import gc
import numpy as np
import pandas as pd

# ---------------------------------------------------------
# 1. Load transaction data
# ---------------------------------------------------------

db = pd.read_csv(
    "../data/raw/train_transaction.csv"
)

print("Transaction:", db.shape)


# ---------------------------------------------------------
# 2. Sort TRANSACTION data before merging
# ---------------------------------------------------------

db = db.sort_values(
    "TransactionDT",
    kind="mergesort"
).reset_index(drop=True)

print("Transaction sorted successfully.")


# ---------------------------------------------------------
# 3. Load identity data
# ---------------------------------------------------------

identity = pd.read_csv(
    "../data/raw/train_identity.csv"
)

print("Identity:", identity.shape)


# ---------------------------------------------------------
# 4. Merge
# ---------------------------------------------------------

data_590k = db.merge(
    identity,
    on="TransactionID",
    how="left",
    sort=False
)

print("Merged data:", data_590k.shape)


# ---------------------------------------------------------
# 5. Free unnecessary objects
# ---------------------------------------------------------

del db
del identity

gc.collect()

print("Memory cleanup completed.")

Transaction: (590540, 394)
Transaction sorted successfully.
Identity: (144233, 41)
Merged data: (590540, 434)
Memory cleanup completed.


In [2]:
# ---------------------------------------------------------
# Time-based features
# ---------------------------------------------------------

data_590k["TransactionHour"] = (
    (data_590k["TransactionDT"] // 3600) % 24
).astype(np.int8)

data_590k["TransactionDay"] = (
    data_590k["TransactionDT"] // (3600 * 24)
).astype(np.int32)

print(
    data_590k[
        [
            "TransactionDT",
            "TransactionHour",
            "TransactionDay"
        ]
    ].head()
)

   TransactionDT  TransactionHour  TransactionDay
0          86400                0               1
1          86401                0               1
2          86469                0               1
3          86499                0               1
4          86506                0               1


In [3]:
data_590k["has_identity"] = (
    data_590k["id_01"].notna()
).astype(np.int8)

In [4]:
data_590k["time_since_previous"] = (
    data_590k["TransactionDT"].diff()
).fillna(0)

In [5]:
data_590k["amount_change"] = (
    data_590k["TransactionAmt"].diff()
).fillna(0)

In [6]:
data_590k["abs_amount_change"] = (
    data_590k["amount_change"].abs()
)

In [7]:
data_590k["card1_frequency"] = (
    data_590k.groupby("card1").cumcount()
)

In [8]:
data_590k["card1_previous_amount_sum"] = (
    data_590k.groupby("card1")["TransactionAmt"]
    .cumsum()
    .shift(1)
)

In [9]:
data_590k["card1_previous_count"] = (
    data_590k.groupby("card1").cumcount()
)

In [10]:
data_590k["card1_avg_previous_amount"] = (
    data_590k["card1_previous_amount_sum"]
    /
    data_590k["card1_previous_count"].replace(
        0,
        np.nan
    )
)

In [11]:
data_590k["card1_avg_previous_amount"] = (
    data_590k["card1_avg_previous_amount"]
    .fillna(
        data_590k["TransactionAmt"].median()
    )
)

In [12]:
data_590k["amount_vs_card_avg"] = (
    data_590k["TransactionAmt"]
    /
    (
        data_590k["card1_avg_previous_amount"]
        + 1e-6
    )
)

In [13]:
data_590k["recent_card_transactions"] = (
    data_590k.groupby("card1")["TransactionDT"]
    .transform(
        lambda x: x.rolling(
            window=10,
            min_periods=1
        ).count()
    )
)

In [14]:
anomaly_features = [
    "TransactionAmt",
    "TransactionHour",
    "TransactionDay",
    "has_identity",
    "time_since_previous",
    "amount_change",
    "abs_amount_change",
    "card1_frequency",
    "card1_avg_previous_amount",
    "amount_vs_card_avg",
    "recent_card_transactions"
]

In [15]:
normal_590k = data_590k[
    data_590k["isFraud"] == 0
].copy()

print(
    "Normal 590K:",
    normal_590k.shape
)

Normal 590K: (569877, 446)


In [17]:
X_normal_590k = normal_590k[
    anomaly_features
].astype(np.float32)

In [18]:
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest

anomaly_scaler_590k = StandardScaler()

X_normal_scaled_590k = (
    anomaly_scaler_590k
    .fit_transform(X_normal_590k)
    .astype(np.float32)
)

In [19]:
isolation_forest_590k = IsolationForest(
    n_estimators=200,
    contamination=0.03,
    random_state=42,
    n_jobs=-1
)

isolation_forest_590k.fit(
    X_normal_scaled_590k
)

print("590K Isolation Forest trained.")

590K Isolation Forest trained.


In [20]:
X_all_590k = data_590k[
    anomaly_features
].astype(np.float32)

X_all_scaled_590k = (
    anomaly_scaler_590k
    .transform(X_all_590k)
    .astype(np.float32)
)

In [21]:
raw_score_590k = (
    isolation_forest_590k
    .decision_function(X_all_scaled_590k)
)

In [22]:
min_score = raw_score_590k.min()
max_score = raw_score_590k.max()

data_590k["anomaly_score_v2"] = (
    (max_score - raw_score_590k)
    /
    (max_score - min_score)
) * 100

In [23]:
split_2 = int(
    len(data_590k) * 0.85
)

test_data_590k = data_590k.iloc[
    split_2:
].copy()

print(
    "Anomaly test:",
    test_data_590k.shape
)

Anomaly test: (88581, 447)


In [24]:
# ============================================================
# 590K CHRONOLOGICAL SPLIT
# ============================================================
split_1 = int(len(data_590k) * 0.70)
split_2 = int(len(data_590k) * 0.85)

train_data_590k = data_590k.iloc[:split_1].copy()
val_data_590k = data_590k.iloc[split_1:split_2].copy()
test_data_590k = data_590k.iloc[split_2:].copy()

print("Train:", train_data_590k.shape)
print("Validation:", val_data_590k.shape)
print("Test:", test_data_590k.shape)

Train: (413378, 447)
Validation: (88581, 447)
Test: (88581, 447)


In [25]:
# ============================================================
# X / y
# ============================================================

X_train_raw = train_data_590k.drop(
    columns=["isFraud", "TransactionID"]
)

X_val_raw = val_data_590k.drop(
    columns=["isFraud", "TransactionID"]
)

X_test_raw = test_data_590k.drop(
    columns=["isFraud", "TransactionID"]
)

y_train_590k = train_data_590k["isFraud"].copy()
y_val_590k = val_data_590k["isFraud"].copy()
y_test_590k = test_data_590k["isFraud"].copy()
print("X_train:", X_train_raw.shape)
print("X_val:", X_val_raw.shape)
print("X_test:", X_test_raw.shape)

print("y_train:", y_train_590k.shape)
print("y_val:", y_val_590k.shape)
print("y_test:", y_test_590k.shape)

X_train: (413378, 445)
X_val: (88581, 445)
X_test: (88581, 445)
y_train: (413378,)
y_val: (88581,)
y_test: (88581,)


In [26]:
# ============================================================
# REMOVE SPARSE FEATURES
# ============================================================

sparse_columns_590k = [
    col
    for col in X_train_raw.columns
    if X_train_raw[col].isnull().mean() > 0.90
]

print("Sparse columns:", len(sparse_columns_590k))
print(sparse_columns_590k)

X_train_raw = X_train_raw.drop(
    columns=sparse_columns_590k
)

X_val_raw = X_val_raw.drop(
    columns=sparse_columns_590k
)

X_test_raw = X_test_raw.drop(
    columns=sparse_columns_590k
)

Sparse columns: 12
['dist2', 'D7', 'id_07', 'id_08', 'id_18', 'id_21', 'id_22', 'id_23', 'id_24', 'id_25', 'id_26', 'id_27']


In [27]:
categorical_columns_590k = X_train_raw.select_dtypes(
    include=["object"]
).columns.tolist()

numerical_columns_590k = X_train_raw.select_dtypes(
    include=["int64", "int32", "int8", "float64", "float32"]
).columns.tolist()

print("Categorical:", len(categorical_columns_590k))
print("Numerical:", len(numerical_columns_590k))

Categorical: 29
Numerical: 404


In [28]:
train_medians_590k = (
    X_train_raw[numerical_columns_590k]
    .median()
)

X_train_raw[numerical_columns_590k] = (
    X_train_raw[numerical_columns_590k]
    .fillna(train_medians_590k)
)

X_val_raw[numerical_columns_590k] = (
    X_val_raw[numerical_columns_590k]
    .fillna(train_medians_590k)
)

X_test_raw[numerical_columns_590k] = (
    X_test_raw[numerical_columns_590k]
    .fillna(train_medians_590k)
)

In [29]:
X_train_raw[categorical_columns_590k] = (
    X_train_raw[categorical_columns_590k]
    .fillna("Unknown")
)

X_val_raw[categorical_columns_590k] = (
    X_val_raw[categorical_columns_590k]
    .fillna("Unknown")
)

X_test_raw[categorical_columns_590k] = (
    X_test_raw[categorical_columns_590k]
    .fillna("Unknown")
)

In [30]:
high_cardinality_590k = [
    col
    for col in categorical_columns_590k
    if X_train_raw[col].nunique() > 100
]

print(
    "High-cardinality columns:",
    high_cardinality_590k
)

High-cardinality columns: ['id_31', 'id_33', 'DeviceInfo']


In [31]:
X_train_raw = X_train_raw.drop(
    columns=high_cardinality_590k
)

X_val_raw = X_val_raw.drop(
    columns=high_cardinality_590k
)

X_test_raw = X_test_raw.drop(
    columns=high_cardinality_590k
)

In [32]:
categorical_columns_590k = X_train_raw.select_dtypes(
    include=["object"]
).columns.tolist()

X_train_590k = pd.get_dummies(
    X_train_raw,
    columns=categorical_columns_590k,
    dummy_na=False
)

X_val_590k = pd.get_dummies(
    X_val_raw,
    columns=categorical_columns_590k,
    dummy_na=False
)

X_test_590k = pd.get_dummies(
    X_test_raw,
    columns=categorical_columns_590k,
    dummy_na=False
)

In [33]:
X_val_590k = X_val_590k.reindex(
    columns=X_train_590k.columns,
    fill_value=0
)

X_test_590k = X_test_590k.reindex(
    columns=X_train_590k.columns,
    fill_value=0
)

In [34]:
X_train_590k = X_train_590k.astype(np.float32)
X_val_590k = X_val_590k.astype(np.float32)
X_test_590k = X_test_590k.astype(np.float32)

print("Final X_train:", X_train_590k.shape)
print("Final X_val:", X_val_590k.shape)
print("Final X_test:", X_test_590k.shape)

Final X_train: (413378, 675)
Final X_val: (88581, 675)
Final X_test: (88581, 675)


In [35]:
from xgboost import XGBClassifier

scale_pos_weight_590k = (
    (y_train_590k == 0).sum()
    /
    (y_train_590k == 1).sum()
)

print(
    "Scale positive weight:",
    scale_pos_weight_590k
)

Scale positive weight: 27.434310083918007


In [36]:
xgb_590k = XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight_590k,
    objective="binary:logistic",
    eval_metric="aucpr",
    tree_method="hist",
    random_state=42,
    n_jobs=-1
)

xgb_590k.fit(
    X_train_590k,
    y_train_590k
)

print("590K XGBoost trained successfully.")

590K XGBoost trained successfully.


In [37]:
xgb_test_prob_590k = (
    xgb_590k.predict_proba(
        X_test_590k
    )[:, 1]
)

print(
    "XGBoost test predictions:",
    len(xgb_test_prob_590k)
)

XGBoost test predictions: 88581


In [38]:
anomaly_risk_590k = (
    test_data_590k[
        "anomaly_score_v2"
    ].to_numpy()
)

y_test_fusion_590k = (
    test_data_590k[
        "isFraud"
    ].to_numpy()
)

print("XGBoost:", len(xgb_test_prob_590k))
print("Anomaly:", len(anomaly_risk_590k))
print("Target:", len(y_test_fusion_590k))

XGBoost: 88581
Anomaly: 88581
Target: 88581


In [39]:
xgb_risk_590k = (
    xgb_test_prob_590k * 100
)

fusion_risk_590k = (
    0.70 * xgb_risk_590k
    +
    0.30 * anomaly_risk_590k
)

print(
    pd.Series(fusion_risk_590k).describe()
)

count    88581.000000
mean        18.780301
std         15.094033
min          1.201678
25%          8.523991
50%         13.966907
75%         23.446885
max         94.594976
dtype: float64


In [40]:
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score
)

fusion_roc_auc_590k = roc_auc_score(
    y_test_fusion_590k,
    fusion_risk_590k
)

fusion_pr_auc_590k = average_precision_score(
    y_test_fusion_590k,
    fusion_risk_590k
)

print("=" * 60)
print("RazorShield 590K Risk Fusion")
print("=" * 60)

print(
    "Fusion ROC-AUC:",
    round(fusion_roc_auc_590k, 4)
)

print(
    "Fusion PR-AUC:",
    round(fusion_pr_auc_590k, 4)
)

RazorShield 590K Risk Fusion
Fusion ROC-AUC: 0.8972
Fusion PR-AUC: 0.484


In [41]:
# ============================================================
# SAVE FINAL RISK SCORES
# ============================================================

final_test_results = pd.DataFrame({
    "isFraud": y_test_fusion_590k,
    "xgb_risk": xgb_risk_590k,
    "anomaly_risk": anomaly_risk_590k,
    "fusion_risk": fusion_risk_590k
})

final_test_results.to_parquet(
    "../data/processed/razorshield_final_test_results.parquet",
    index=False
)

print("Final test results saved.")
print("Shape:", final_test_results.shape)

Final test results saved.
Shape: (88581, 4)


In [42]:
import os
import joblib

os.makedirs("../models", exist_ok=True)

# ============================================================
# 1. SAVE XGBOOST 590K
# ============================================================

joblib.dump(
    xgb_590k,
    "../models/razorshield_xgboost_590k.pkl"
)

# ============================================================
# 2. SAVE XGBOOST FEATURE COLUMNS
# ============================================================

joblib.dump(
    X_train_590k.columns.tolist(),
    "../models/razorshield_xgboost_590k_features.pkl"
)

# ============================================================
# 3. SAVE ANOMALY SCALER
# ============================================================

joblib.dump(
    anomaly_scaler_590k,
    "../models/razorshield_anomaly_scaler_590k.pkl"
)

# ============================================================
# 4. SAVE ISOLATION FOREST
# ============================================================

joblib.dump(
    isolation_forest_590k,
    "../models/razorshield_isolation_forest_590k.pkl"
)

# ============================================================
# 5. SAVE ANOMALY FEATURE LIST
# ============================================================

joblib.dump(
    anomaly_features,
    "../models/razorshield_anomaly_features_590k.pkl"
)

print("=" * 60)
print("ALL RAZORSHIELD 590K MODELS SAVED")
print("=" * 60)

ALL RAZORSHIELD 590K MODELS SAVED
